In [1]:
# Imports, device, and reproducibility

from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from torch import nn

from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch_geometric.nn import SAGEConv, BatchNorm, global_mean_pool

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from scipy.sparse import issparse

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cpu


In [2]:
# Load the concrete dataset

CSV_PATH = "Concrete_Data.csv"
TARGET_COL = "Concrete compressive strength(MPa, megapascals) "

df = pd.read_csv(CSV_PATH)
assert TARGET_COL in df.columns, f"Target missing; check columns: {df.columns.tolist()}"

X_df = df.drop(columns=[TARGET_COL]).copy()   
y_raw = df[TARGET_COL].to_numpy(dtype=float).reshape(-1, 1)

print("Dataset:", df.shape)
print("Features:", X_df.shape)


Dataset: (1030, 9)
Features: (1030, 8)


In [11]:
# split

N = len(df)
indices = np.arange(N)

train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=42
)

val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=42
)

assert len(set(train_idx) & set(val_idx)) == 0
assert len(set(train_idx) & set(test_idx)) == 0
assert len(set(val_idx) & set(test_idx)) == 0

print(
    f"Split -> Train {len(train_idx)}, "
    f"Val {len(val_idx)}, "
    f"Test {len(test_idx)}"
)

Split -> Train 721, Val 154, Test 155


In [4]:
# Fit preprocessing and target scaling on TRAINING DATA ONLY

num_cols = X_df.select_dtypes(include=[np.number]).columns.tolist()
cat_cols = [c for c in X_df.columns if c not in num_cols]

def build_ohe():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

transformers = [("num", numeric_pipe, num_cols)]

if cat_cols:
    categorical_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", build_ohe())
    ])
    transformers.append(("cat", categorical_pipe, cat_cols))

preprocess = ColumnTransformer(
    transformers=transformers,
    remainder="drop",
    sparse_threshold=0.3
)

X_train = preprocess.fit_transform(X_df.iloc[train_idx])
X_val   = preprocess.transform(X_df.iloc[val_idx])
X_test  = preprocess.transform(X_df.iloc[test_idx])

def dense_float(x):
    return np.asarray(x.toarray() if issparse(x) else x, dtype=np.float32)

X_train, X_val, X_test = map(dense_float, [X_train, X_val, X_test])

# Target scaler is also fitted on training targets only.
y_scaler = StandardScaler().fit(y_raw[train_idx])
y_train = y_scaler.transform(y_raw[train_idx]).ravel().astype(np.float32)
y_val   = y_scaler.transform(y_raw[val_idx]).ravel().astype(np.float32)
y_test  = y_scaler.transform(y_raw[test_idx]).ravel().astype(np.float32)

print("Scaled features:", X_train.shape, X_val.shape, X_test.shape)


Scaled features: (743, 8) (132, 8) (155, 8)


In [5]:
# Build STRICTLY INDUCTIVE local graphs

k = 3

def fully_connected_edges(n):
    ii, jj = np.triu_indices(n, k=1)
    return np.vstack([
        np.concatenate([ii, jj]),
        np.concatenate([jj, ii])
    ])

train_knn = NearestNeighbors(n_neighbors=k+1, metric="euclidean")
train_knn.fit(X_train)

def build_training_graphs():
    graphs = []
    neigh_matrix = train_knn.kneighbors(
        X_train, n_neighbors=k+1, return_distance=False
    )

    for center_local in range(len(X_train)):
        row = neigh_matrix[center_local]

        
        neigh = row[row != center_local][:k]
        if len(neigh) != k:
            raise RuntimeError(f"Could not obtain {k} non-self train neighbors for center {center_local}")

        
        local_X = np.vstack([X_train[center_local:center_local+1], X_train[neigh]])
        edge_index = torch.tensor(
            fully_connected_edges(k + 1), dtype=torch.long
        )
        y_i = torch.tensor([y_train[center_local]], dtype=torch.float32)

        graphs.append(
            Data(
                x=torch.tensor(local_X, dtype=torch.float32),
                edge_index=edge_index,
                y=y_i
            )
        )
    return graphs

def build_query_graphs(X_query, y_query):
    graphs = []

    neigh_matrix = train_knn.kneighbors(
        X_query, n_neighbors=k, return_distance=False
    )

    for q in range(len(X_query)):
        neigh = neigh_matrix[q]

        local_X = np.vstack([X_query[q:q+1], X_train[neigh]])
        edge_index = torch.tensor(
            fully_connected_edges(k + 1), dtype=torch.long
        )
        y_i = torch.tensor([y_query[q]], dtype=torch.float32)

        graphs.append(
            Data(
                x=torch.tensor(local_X, dtype=torch.float32),
                edge_index=edge_index,
                y=y_i
            )
        )
    return graphs

train_graphs = build_training_graphs()
val_graphs   = build_query_graphs(X_val, y_val)

print("Training graphs:", len(train_graphs))
print("Validation graphs:", len(val_graphs))
print("Example training graph:", train_graphs[0])
print("Each local graph nodes:", train_graphs[0].num_nodes)


Training graphs: 743
Validation graphs: 132
Example training graph: Data(x=[4, 8], edge_index=[2, 12], y=[1])
Each local graph nodes: 4


In [6]:
# graph-level GraphSAGE architecture

class GraphLevelSAGENet(nn.Module):
    def __init__(self, in_channels, hidden=128, layers=3, dropout=0.25):
        super().__init__()

        self.convs = nn.ModuleList()
        self.bns   = nn.ModuleList()

        self.convs.append(SAGEConv(in_channels, hidden))
        self.bns.append(BatchNorm(hidden))

        for _ in range(layers - 2):
            self.convs.append(SAGEConv(hidden, hidden))
            self.bns.append(BatchNorm(hidden))

        self.dropout = dropout
        self.head = nn.Linear(hidden, 1)

    def forward(self, x, edge_index, batch):
        for conv, bn in zip(self.convs, self.bns):
            x = conv(x, edge_index)
            x = bn(x)
            x = F.relu(x)
            x = F.dropout(x, p=self.dropout, training=self.training)

        g = global_mean_pool(x, batch)
        out = self.head(g).view(-1)
        return out

set_seed(42)

in_dim = train_graphs[0].num_features
model = GraphLevelSAGENet(
    in_channels=in_dim,
    hidden=128,
    layers=3,
    dropout=0.25
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-4
)
criterion = nn.MSELoss()

print("Input features:", in_dim)
print("Number of SAGEConv layers actually created:", len(model.convs))


Input features: 8
Number of SAGEConv layers actually created: 2


In [7]:
# DataLoaders
batch_size = 64

train_loader = DataLoader(
    train_graphs, batch_size=batch_size, shuffle=True
)
val_loader = DataLoader(
    val_graphs, batch_size=batch_size, shuffle=False
)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))


Train batches: 12
Val batches: 3


In [8]:
# Train using TRAIN only; select checkpoint using VALIDATION only
# TEST data are NOT evaluated or monitored during training.

CKPT_DIR = Path.cwd() / "artifacts"
CKPT_DIR.mkdir(parents=True, exist_ok=True)
best_path = CKPT_DIR / "gnn_graph_inductive_best.pt"

best_val = float("inf")
best_epoch = None
patience = 30
bad = 0
min_delta = 1e-4
max_epochs = 1000
history = []

def run_one_epoch(loader, train=True):
    model.train() if train else model.eval()

    running = 0.0
    n = 0

    with torch.set_grad_enabled(train):
        for batch in loader:
            batch = batch.to(device)

            if train:
                optimizer.zero_grad()

            out = model(batch.x, batch.edge_index, batch.batch)
            loss = criterion(out, batch.y.view(-1))

            if train:
                loss.backward()
                optimizer.step()

            running += loss.item() * batch.num_graphs
            n += batch.num_graphs

    return running / max(n, 1)

y_std = float(y_scaler.scale_[0])

for epoch in range(1, max_epochs + 1):
    tr_loss = run_one_epoch(train_loader, train=True)
    val_loss = run_one_epoch(val_loader, train=False)

    tr_rmse_mpa = np.sqrt(tr_loss) * y_std
    val_rmse_mpa = np.sqrt(val_loss) * y_std

    history.append((epoch, tr_loss, val_loss, tr_rmse_mpa, val_rmse_mpa))

    if val_loss < best_val - min_delta:
        best_val = val_loss
        best_epoch = epoch
        bad = 0
        torch.save(model.state_dict(), best_path)
    else:
        bad += 1

    if epoch % 25 == 0 or bad == 0:
        print(
            f"Epoch {epoch:4d} | "
            f"train RMSE {tr_rmse_mpa:.4f} MPa | "
            f"val RMSE {val_rmse_mpa:.4f} MPa"
        )

    if bad >= patience:
        print(f"Early stop at epoch {epoch}. Best epoch {best_epoch}.")
        break

model.load_state_dict(torch.load(best_path, map_location=device))
print("Loaded best checkpoint:", best_path)
print("Best validation epoch:", best_epoch)


Epoch    1 | train RMSE 13.2724 MPa | val RMSE 13.5130 MPa
Epoch    2 | train RMSE 10.4139 MPa | val RMSE 11.7594 MPa
Epoch    3 | train RMSE 9.8730 MPa | val RMSE 10.9678 MPa
Epoch    4 | train RMSE 9.2293 MPa | val RMSE 10.7582 MPa
Epoch    5 | train RMSE 8.9598 MPa | val RMSE 10.2572 MPa
Epoch    7 | train RMSE 8.7587 MPa | val RMSE 10.0548 MPa
Epoch    8 | train RMSE 8.8245 MPa | val RMSE 10.0272 MPa
Epoch    9 | train RMSE 9.2187 MPa | val RMSE 9.8527 MPa
Epoch   10 | train RMSE 8.7946 MPa | val RMSE 9.7863 MPa
Epoch   11 | train RMSE 8.5525 MPa | val RMSE 9.6782 MPa
Epoch   12 | train RMSE 8.4365 MPa | val RMSE 9.6235 MPa
Epoch   14 | train RMSE 8.4891 MPa | val RMSE 9.1251 MPa
Epoch   23 | train RMSE 8.0001 MPa | val RMSE 9.0747 MPa
Epoch   25 | train RMSE 7.9566 MPa | val RMSE 9.0045 MPa
Epoch   28 | train RMSE 8.2461 MPa | val RMSE 8.9853 MPa
Epoch   30 | train RMSE 7.9685 MPa | val RMSE 8.9673 MPa
Epoch   32 | train RMSE 8.1218 MPa | val RMSE 8.9080 MPa
Epoch   38 | train RMS

In [9]:
# Build the held-out TEST graphs ONLY AFTER model selection
test_graphs = build_query_graphs(X_test, y_test)
test_loader = DataLoader(
    test_graphs, batch_size=batch_size, shuffle=False
)

print("Test graphs:", len(test_graphs))
print("Test graph example:", test_graphs[0])


Test graphs: 155
Test graph example: Data(x=[4, 8], edge_index=[2, 12], y=[1])


In [10]:
# Final metrics in MPa
@torch.no_grad()
def collect_true_pred(loader):
    model.eval()
    preds, trues = [], []

    for batch in loader:
        batch = batch.to(device)
        out = model(batch.x, batch.edge_index, batch.batch)

        preds.append(out.detach().cpu().numpy().reshape(-1, 1))
        trues.append(batch.y.detach().cpu().numpy().reshape(-1, 1))

    preds = np.vstack(preds)
    trues = np.vstack(trues)

    y_true = y_scaler.inverse_transform(trues).ravel()
    y_pred = y_scaler.inverse_transform(preds).ravel()
    return y_true, y_pred

def metrics(y_true, y_pred):
    rmse = mean_squared_error(y_true, y_pred) ** 0.5
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    # Concrete strengths are positive in this dataset.
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    return rmse, mae, r2, mape

results = {}
for name, loader in [
    ("Train", train_loader),
    ("Val", val_loader),
    ("Test", test_loader),
]:
    yt, yp = collect_true_pred(loader)
    rmse, mae, r2, mape = metrics(yt, yp)
    results[name] = {
        "RMSE_MPa": rmse,
        "MAE_MPa": mae,
        "R2": r2,
        "MAPE_pct": mape,
    }
    print(
        f"{name:5s} | "
        f"RMSE {rmse:.4f} | "
        f"MAE {mae:.4f} | "
        f"R² {r2:.4f} | "
        f"MAPE {mape:.2f}%"
    )

print("\nBest validation epoch:", best_epoch)


Train | RMSE 5.9632 | MAE 4.4198 | R² 0.8678 | MAPE 17.33%
Val   | RMSE 8.0136 | MAE 5.9549 | R² 0.8176 | MAPE 23.68%
Test  | RMSE 6.9304 | MAE 5.3307 | R² 0.8166 | MAPE 19.89%

Best validation epoch: 188


In [12]:
import sys
import torch
import torch_geometric
import sklearn

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("PyTorch Geometric:", torch_geometric.__version__)
print("scikit-learn:", sklearn.__version__)

Python: 3.12.7 | packaged by Anaconda, Inc. | (main, Oct  4 2024, 13:17:27) [MSC v.1929 64 bit (AMD64)]
PyTorch: 2.7.1+cpu
PyTorch Geometric: 2.8.0.post1
scikit-learn: 1.5.1
